# Module 5.2: Unified Multi-Node Observability & Health Monitoring with Uptime Kuma

## Homelab Expansion (Chapter 2) - Step 5: Core Raspberry Pi Infrastructure Services

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why an observability and alerting service (**Uptime Kuma**) should run on the dedicated Tier-0 core node (`homelab-pi`)—physically separate from the app/GPU server (`homelab-gpu`) it monitors.
2. Deploy **Uptime Kuma** on `homelab-pi` (`10.0.0.200:3001`) with persistent local storage under `~/homelab/data/uptime-kuma`.
3. Configure real-time HTTP, DNS, and ICMP Ping monitors across both `homelab-pi` and `homelab-gpu` (including the migrated full Immich API at `http://10.0.0.201:2283/api/server/ping`).
4. Review the complete two-node homelab operational checklist.

## 5.2.1 Conceptual Background: Watching the Watchmen in a Multi-Node Cluster

Now that your homelab cleanly separates **Tier-0 Network & Observability (`homelab-pi` at `10.0.0.200`)** from **Applications & GPU Compute (`homelab-gpu` at `10.0.0.201`)** across your `USW-Enterprise-8-PoE` switch, you need an independent pane of glass to verify that every host, port, and container is healthy.

```mermaid
flowchart LR
    Kuma["Uptime Kuma on homelab-pi<br>http://10.0.0.200:3001"]
    
    Kuma -->|"ICMP Ping"| Laptop["Node 2: homelab-gpu (10.0.0.201)"]
    Kuma -->|"HTTP GET :2283/api/server/ping"| Immich["Immich Full Stack (10.0.0.201:2283)"]
    Kuma -->|"DNS Query :53 (github.com)"| Pihole["Pi-hole v6 DNS (10.0.0.200:53)"]
    Kuma -->|"HTTP GET :8080/admin"| PiholeUI["Pi-hole v6 Admin UI (10.0.0.200:8080)"]
```

By hosting **Uptime Kuma** on `homelab-pi`:
- It consumes only **~65–110 MB of RAM** and `<1%` CPU.
- If `homelab-gpu` ever goes offline, reboots after a kernel/NVIDIA driver update, or Immich stops responding on `:2283`, `homelab-pi` stays online to record the exact outage timestamp, track historical latency graphs across the switch, and optionally push an instant alert to your phone via Discord, Telegram, Ntfy, or Email.

## 5.2.2 Step-by-Step Laboratory Guide

### Task 5.2.1: Deploy Uptime Kuma on `homelab-pi`
Connect to your Raspberry Pi (`ssh homelab-pi`) and create the Uptime Kuma stack directory under `~/homelab/`:
```bash
mkdir -p ~/homelab/stacks/uptime-kuma
mkdir -p ~/homelab/data/uptime-kuma
cd ~/homelab/stacks/uptime-kuma
```

Create the `docker-compose.yml` file:
```bash
cat << 'EOT' > ~/homelab/stacks/uptime-kuma/docker-compose.yml
name: uptime_kuma

services:
  uptime-kuma:
    container_name: uptime_kuma
    image: louislam/uptime-kuma:1
    ports:
      - "3001:3001"
    volumes:
      - /home/pi/homelab/data/uptime-kuma:/app/data
      # Optional: Mount docker.sock read-only to monitor local Pi containers directly!
      - /var/run/docker.sock:/var/run/docker.sock:ro
    environment:
      - TZ=America/New_York
    restart: unless-stopped
EOT
```

Launch the container and verify it is running:
```bash
docker compose up -d
docker compose ps
```

### Task 5.2.2: Configure Multi-Node Health Monitors in Uptime Kuma
1. Open your browser and navigate to:
   ```text
   http://10.0.0.200:3001
   ```
   (or `http://homelab-pi:3001` over Tailscale).
2. Create your administrator account on the initial setup screen.
3. Click **+ Add New Monitor** to register the four foundational pillars of your two-node homelab:

| Monitor Friendly Name | Monitor Type | Target URL / Hostname | Port / Expected Keyword | Heartbeat Interval |
| :--- | :--- | :--- | :--- | :---: |
| **1. Node 2 (`homelab-gpu`) Host Ping** | `Ping` | `10.0.0.201` | Sub-ms LAN ICMP ping | `60s` |
| **2. Immich Web & Core API (`homelab-gpu`)** | `HTTP(s) - Keyword` | `http://10.0.0.201:2283/api/server/ping` | Keyword: `pong` | `60s` |
| **3. Pi-hole v6 DNS Resolver (`homelab-pi`)** | `DNS` | `10.0.0.200` (Resolve: `github.com`, Record: `A`) | Port `53` | `60s` |
| **4. Pi-hole v6 Admin UI (`homelab-pi`)** | `HTTP(s)` | `http://10.0.0.200:8080/admin` | HTTP `200`/`302` | `60s` |

### 🖼️ Task 5.2.2 Verification: Unified Multi-Node Uptime Kuma Dashboard

**Verification Objective:** Capture a screenshot of your Uptime Kuma dashboard (`http://10.0.0.200:3001`) showing `100%` green status bars across both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Chapter 2 Final Checkpoint & Reflection

Congratulations on expanding your single-board Raspberry Pi NAS into a cleanly isolated, GPU-accelerated two-node homelab! Confirm your final Chapter 2 milestones below:

- [ ] **Hardware, Thermal & Internal NVMe Migration:** 2019 ROG Zephyrus Li-Po battery removed, header insulated with Kapton tape, CPU/GPU repasted & VRM putty refreshed, and your **WD Blue SN5000** installed directly into internal **M.2 Slot 2** (backed by an external UPS).
- [ ] **Headless OS & Pi Cleanup:** Unlicensed Windows 11 wiped from Slot 1 for **Ubuntu Server 24.04 LTS (`x86_64`)**, Slot 2 mounted at `/mnt/homelab_storage`, `HandleLidSwitch=ignore` & `consoleblank=60` active, and **`homelab-pi` cleanly stripped of old Immich/NVMe `/etc/fstab` entries** and verified rebooting standalone.
- [ ] **Unified `USW-Enterprise-8-PoE` Switch Topology:** Both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`) connected to `USW-Enterprise-8-PoE` at wire speed and joined to your Tailscale mesh.
- [ ] **Full Immich Migration (`CUDA` + `NVENC`):** Complete 4-container Immich stack running on `homelab-gpu:2283` with Chapter 1 PostgreSQL database restored, NVIDIA **CUDA** ML active, and **NVENC** hardware video transcoding enabled.
- [ ] **Dedicated Tier-0 Raspberry Pi:** **Pi-hole v6** (`:53` / `:8080`) providing LAN & global Tailscale ad-blocking, and **Uptime Kuma** (`:3001`) independently monitoring both nodes and all core services.

Ready for Enterprise Storage & UniFi Ecosystem Management? Proceed to [Chapter 3: UniFi UNAS 2 Storage & Enterprise Networking](../../03_unifi_nas_and_enterprise_networking/README.md)!